In [11]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import wandb

wandb.init(project='mini-llm', name='SFT')

if torch.cuda.is_available():
    device='cuda'
    print(f"Using CUDA GPU: {torch.cuda.get_device_name()}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory /1e9} GB")

elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
    device='mps'
    print("Using Apple MPS")
else:
    device = "cpu"
    print("Using CPU - you will need to use a GPU to train models")

model_name = "Qwen/Qwen2.5-0.5B"
new_model_name = "Qwen2.5-0.5B-SFT"


eval/entropy,█▄▂▃▂▂▁▁▁▁▁▁▁▁▁▁▁
eval/loss,█▅▄▄▃▂▂▂▁▁▁▁▁▁▁▁▁
eval/mean_token_accuracy,▁▅▆▆▇▇▇▇█████████
eval/num_tokens,▁▁▂▂▃▃▄▄▅▅▅▆▆▇▇██
eval/runtime,▇▆▄▅██▇█▇▇▁██▇█▇█
eval/samples_per_second,▂▃▅▄▁▁▂▁▂▂█▁▁▂▁▂▁
eval/steps_per_second,▂▃▅▄▁▁▂▁▂▂█▁▁▂▁▂▁
train/entropy,█▅▄▄▄▂▂▁▂▁▂▂▂▂▂▁▁▂▂▂▂▁▁▂▁▁▁▂▁▁▁▁▂▁▁▂▁▂▂▂
train/epoch,▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇█
+5,...


Using CUDA GPU: NVIDIA GeForce RTX 5070 Ti
GPU memory: 16.587751424 GB


In [12]:
random_seed = 42
eval_size = 200
dataset_size = 7000

torch.manual_seed(random_seed)

In [13]:
model = AutoModelForCausalLM.from_pretrained(model_name,
                                             torch_dtype=torch.bfloat16)
tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.padding_side = "right"

tokenizer.add_special_tokens({"pad_token": "<|pad|>"})
model.resize_token_embeddings(len(tokenizer))

train_dataset = load_dataset("HuggingFaceTB/smoltalk2",
    "SFT",
    split="OpenHermes_2.5_no_think"
    ).select(range(dataset_size))


print("Dataset size", len(train_dataset))
print("Example", train_dataset[0])

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 290/290 [00:00<00:00, 25476.46it/s]


Dataset size 7000
Example {'messages': [{'content': 'What large island country located off the southeastern coast of Africa is home to many unique species like lemurs?', 'role': 'user'}, {'content': "Ah, you're talking about the land of lemurs and baobabs! That's Madagascar, my friend. It's a world all its own with species that can't be found anywhere else on the planet. If I were to mix a cocktail inspired by it, I'd probably go for something exotic and unique too - maybe a rum base with some tropical fruits. Now wouldn't that take your taste buds on an adventure?", 'role': 'assistant'}], 'chat_template_kwargs': {'custom_instructions': '', 'enable_thinking': False, 'python_tools': [], 'xml_tools': []}, 'source': 'OpenHermes-2.5'}


In [14]:

train_dataset = train_dataset.shuffle(seed=random_seed)
val_dataset = train_dataset.select(range(eval_size))
train_dataset = train_dataset.select(range(eval_size, dataset_size))

In [15]:
import json
import os

eval_prompts = [[{"role": "user", "content": ex["messages"][0]["content"]}] for ex in val_dataset]

if not(os.path.isfile("eval_prompts.json")):
    with open("eval_prompts.json", "w") as f:
        json.dump(eval_prompts, f, ensure_ascii=False, indent=2)


In [16]:
sample = train_dataset.select(range(100))
sample[10]

{'messages': [{'content': 'Bayern is the German name for which region of Germany????',
   'role': 'user'},
  {'content': 'The German name "Bayern" translates to "Bavaria" in English. \n\nTo provide this information, I utilized my programmed knowledge base of various global data, which includes information on geographical areas, regions, and places around the world. I also have direct access to a wide array of databases and can retrieve and deliver accurate data in realtime. In this case, the translation and association between Bayern and Bavaria is commonly known and a part of general knowledge, hence the immediate response.',
   'role': 'assistant'}],
 'chat_template_kwargs': {'custom_instructions': '',
  'enable_thinking': False,
  'python_tools': [],
  'xml_tools': []},
 'source': 'OpenHermes-2.5'}

In [17]:
from trl import SFTConfig, SFTTrainer

training_config = SFTConfig(
    output_dir=f"./{new_model_name}",
    max_length=2048,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    bf16=True,
    num_train_epochs=1,
    max_steps=-1,

    warmup_steps=50,
    weight_decay=0.01,
    optim="adamw_torch_fused",

    logging_steps=10,
    save_steps=100,
    save_total_limit=2,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    dataloader_num_workers=0,

    push_to_hub=False,
    report_to=["wandb"],
    run_name=f"{new_model_name}-training",
    eval_strategy="steps",
    eval_steps=100,
)

print("Training configuration set!")
print(f"Effective batch size: {training_config.per_device_train_batch_size * training_config.gradient_accumulation_steps}")

Training configuration set!
Effective batch size: 4


In [18]:

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    processing_class=tokenizer,
    args=training_config,
    eval_dataset=val_dataset
)

trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151665}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.900566,1.729531,1.860433,83895.000000,0.618183
200,1.529262,1.692392,1.751887,170235.000000,0.629686
300,1.322500,1.673096,1.706202,252989.000000,0.631936
400,1.619289,1.668444,1.722702,342503.000000,0.631281
500,1.596138,1.659043,1.711881,430892.000000,0.632696
600,1.687136,1.652610,1.700398,522425.000000,0.633051
700,1.694858,1.650132,1.678760,601729.000000,0.634303
800,1.893683,1.644860,1.680454,699482.000000,0.634562
900,1.636468,1.638926,1.685893,791846.000000,0.636006
1000,1.818796,1.637782,1.681935,877921.000000,0.636630


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  4.23it/s]


TrainOutput(global_step=1700, training_loss=1.6284527329837575, metrics={'train_runtime': 452.8599, 'train_samples_per_second': 15.016, 'train_steps_per_second': 3.754, 'total_flos': 3200792332438272.0, 'train_loss': 1.6284527329837575, 'epoch': 1.0})

In [19]:
trainer.save_model(f"./{new_model_name}/final")
tokenizer.save_pretrained(f"./{new_model_name}/final")
print("Model saved")

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.37it/s]


Model saved


In [20]:
import json

with open("eval_prompts.json") as f:
    eval_prompts = json.load(f)

model.eval()
results = []

im_end_id = tokenizer.convert_tokens_to_ids("<|im_end|>")
eos_ids = [tokenizer.eos_token_id, im_end_id]

for i, messages in enumerate(eval_prompts[:20]):
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            do_sample=True,
            eos_token_id=eos_ids,
            pad_token_id=tokenizer.pad_token_id,
        )
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    results.append({"prompt": messages[0]["content"], "response": response})
    print(f"--- Prompt {i} ---")
    print(f"Q: {messages[0]['content'][:100]}")
    print(f"A: {response[:300]}")
    print()

with open("sft_generations.json", "w") as f:
    json.dump(results, f, ensure_ascii=False, indent=2)
print(f"Saved {len(results)} generations to sft_generations.json")

--- Prompt 0 ---
Q: Mountain Jews or Caucasus Jews also known as Juhuro, Juvuro, Juhuri, Juwuri, Juhurim, Kavkazi Jews o
A: According to the article, women in the Mountain Jewish community learned Hebrew, which was the language of instruction at newly founded elementary schools attended by both Mountain Jewish boys and girls.

--- Prompt 1 ---
Q: Alice Geraldine Farrar (February 28, 1882 – March 11, 1967) was an American soprano opera singer and
A: B). It's impossible to say.

The provided paragraph does not mention any information about Alice Geraldine Farrar's involvement in either her films or as an opera singer. They are separate professions, and it is not possible to conclude that the paragraph talks about her collaborations or work in fi

--- Prompt 2 ---
Q: What chemical compound, also known as H2O, is essential for all known forms of life and covers about
A: Water

--- Prompt 3 ---
Q: What is the answer: What is the name of the main character in John Osborne's 'Look Back In Ang